In [1]:
!pip install -qU langchain-community langchain-huggingface langgraph peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.7/508.7 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 6.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have j

In [2]:
!pip install faiss-gpu-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 4.0 MB/s eta 0:00:0000:0100:010m


In [3]:
import os

# This is the path based on your 'adapter' notebook output image
adapter_path = "/kaggle/input/notebooks/mohammadalthafnawaz/adapter/mistral_lora_adapter"

if os.path.exists(adapter_path):
    print(f"✅ Path found! Contents: {os.listdir(adapter_path)}")
else:
    print("❌ Path NOT found. Please check if the 'adapter' notebook is added as Input.")

✅ Path found! Contents: ['adapter_model.safetensors', 'adapter_config.json', 'README.md', 'tokenizer.json', 'tokenizer_config.json']


In [4]:
# This is the path based on your 'knowledge-base' notebook output
faiss_path = "/kaggle/input/notebooks/mohammadalthafnawaz/kb-new/chunked_supreme_court_index_final"

if os.path.exists(faiss_path):
    print(f"✅ FAISS Index found! Contents: {os.listdir(faiss_path)}")
else:
    print("❌ FAISS Path NOT found. Please check if the 'knowledge-base' notebook is added as Input.")

✅ FAISS Index found! Contents: ['index.faiss', 'index.pkl']


In [5]:
import re

In [6]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline

# 1. Configuration for T4 GPU (4-bit quantization)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# 2. Setup Paths
model_id = "mistralai/Mistral-7B-v0.1"
# We use the path you verified in Cell 2
adapter_path = "/kaggle/input/notebooks/mohammadalthafnawaz/adapter/mistral_lora_adapter"

# 3. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# 4. Load Base Model
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 5. Snap your Adapter on top
model = PeftModel.from_pretrained(base_model, adapter_path)

# 6. Create the LangChain pipeline
pipe = pipeline(
    "text-generation", 
    model=model, 
    tokenizer=tokenizer, 
    max_new_tokens=256,
    temperature=0.1, # Low temperature for factual legal answers
    do_sample=True
)
llm = HuggingFacePipeline(pipeline=pipe)

print("🚀 Shadow Legal Brain Loaded!")

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


🚀 Shadow Legal Brain Loaded!


In [7]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
import faiss

# Embeddings on GPU
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={'device': 'cuda'}
)

faiss_path = "/kaggle/input/notebooks/mohammadalthafnawaz/kb-new/chunked_supreme_court_index_final"

# Load vectorstore (CPU index initially)
vectorstore = FAISS.load_local(
    faiss_path, 
    embeddings, 
    allow_dangerous_deserialization=True
)

print("Loaded FAISS index (CPU)")

# 🔥 Move FAISS index to GPU
cpu_index = vectorstore.index

res = faiss.StandardGpuResources()
gpu_index = faiss.index_cpu_to_gpu(res, 0, cpu_index)

# Replace index inside vectorstore
vectorstore.index = gpu_index

print("🚀 FAISS index moved to GPU!")

# Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

print("📂 Structured Supreme Court KB Connected with GPU FAISS!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded FAISS index (CPU)
🚀 FAISS index moved to GPU!
📂 Structured Supreme Court KB Connected with GPU FAISS!


In [8]:
import numpy as np

In [9]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, END

# 1. Define the State
class AgentState(TypedDict):
    question: str
    context: str
    answer: str
    iteration: int
    confidence: float
    augmented_query: str

# 2. NODE: Retrieval (Strategic Two-Stage Initial + Fast Loop)
def retrieve_node(state: AgentState):
    iteration = state.get("iteration", 0) + 1
    current_query = state.get("augmented_query") or state["question"]
    
    if iteration == 1:
        # --- STAGE 1: Initial Noisy Retrieval ---
        docs_1 = retriever.invoke(state["question"])
        noisy_text = "\n".join([d.page_content for d in docs_1])
        
        # --- STAGE 2: ADAPTER ALIGNMENT (Mθ) ---
        align_prompt = f"Extract only the key legal reasoning sentences for: {state['question']}\nContext: {noisy_text}"
        reasoning_clues = llm.invoke(align_prompt)
        
        # --- STAGE 3: QUERY AUGMENTATION (q2) ---
        new_query = f"{state['question']} {reasoning_clues}"
        
        # --- STAGE 4: REFINED RETRIEVAL ---
        docs_2 = retriever.invoke(new_query)
        context = "\n\n".join([d.page_content for d in docs_2])
        return {"context": context, "iteration": iteration, "augmented_query": new_query}
    
    else:
        # --- FAST LOOP: Single-Stage Retrieval using q2 ---
        # We don't repeat the adapter alignment to save time/compute
        docs_fast = retriever.invoke(current_query)
        context = "\n\n".join([d.page_content for d in docs_fast])
        return {"context": context, "iteration": iteration}

# 3. NODE: Generation (G)
def generate_node(state: AgentState):
    # 1. Generate Answer
    prompt = f"<s>[INST] Context: {state['context']}\nQuestion: {state['question']} [/INST] Answer:"
    answer = llm.invoke(prompt).split("Answer:")[-1].strip()
    
    # 2. Vector-Based Confidence (No Keywords)
    # We compare the 'vibe' of the answer to the 'vibe' of the context
    ans_enc = np.array(embeddings.embed_query(answer))
    ctx_enc = np.array(embeddings.embed_query(state['context']))
    
    # Calculate Cosine Similarity
    score = np.dot(ans_enc, ctx_enc) / (np.linalg.norm(ans_enc) * np.linalg.norm(ctx_enc))
    
    # Normalize score (usually falls between 0.6 and 0.9 for good matches)
    # We map 0.7+ to "High Confidence"
    return {"answer": answer, "confidence": float(score)}

# 5. ROUTING: The Decision Gate
def confidence_check(state: AgentState):
    if state["confidence"] >= 0.7 or state["iteration"] >= 2:
        return "finalize"
    return "refine" # Triggers the Fast Loop back to Retrieve

# 6. BUILD ARCHITECTURE
builder = StateGraph(AgentState)
builder.add_node("retrieve", retrieve_node)
builder.add_node("generate", generate_node)

builder.set_entry_point("retrieve")
builder.add_edge("retrieve", "generate")
builder.add_conditional_edges(
    "generate", 
    confidence_check, 
    {"finalize": END, "refine": "retrieve"}
)

shadow_agent = builder.compile()
print("🔥 Shadow Agent Compiled: Initial 2-Stage + Efficient 1-Stage Loop Ready!")

🔥 Shadow Agent Compiled: Initial 2-Stage + Efficient 1-Stage Loop Ready!


In [10]:
import gradio as gr
from datetime import datetime

# ==============================
# FUNCTION
# ==============================
def process_legal_query(query, chat_history):

    history_log = "🧠 Initializing...\n"

    initial_state = {
        "question": query,
        "iteration": 0,
        "confidence": 0.0,
        "context": "",
        "answer": "",
        "augmented_query": ""
    }

    yield chat_history, history_log

    final_answer = ""
    final_conf = 0.0

    for output in shadow_agent.stream(initial_state):
        for node_name, state_update in output.items():

            if node_name == "retrieve":
                it = state_update.get("iteration", 1)
                if it == 1:
                    history_log += "📡 Retrieving cases...\n"
                else:
                    history_log += f"🔄 Refining search {it}...\n"

            if node_name == "generate":
                final_answer = state_update.get("answer", "")
                final_conf = state_update.get("confidence", 0.0)

                history_log += f"⚡ Generating (confidence: {final_conf:.2f})\n"

                if final_conf < 0.75:
                    history_log += "⚠️ Improving answer...\n"
                else:
                    history_log += "✅ Final answer ready\n"

        yield chat_history, history_log

    timestamp = datetime.now().strftime("%H:%M:%S")

    chat_history.append((
        f"👤 {query}\n🕒 {timestamp}",
        f"⚖️ {final_answer}"
    ))

    history_log += "\n🏁 Done."

    yield chat_history, history_log


# ==============================
# UI
# ==============================
with gr.Blocks(
    theme=gr.themes.Base(),
    css="""
    body {
        background: linear-gradient(135deg, #020617, #0f172a, #020617);
        font-family: 'Inter', sans-serif;
    }

    .gradio-container {
        background: transparent;
    }

    /* Chat styling */
    .chatbot {
        border-radius: 20px;
        background: rgba(15, 23, 42, 0.7);
        backdrop-filter: blur(20px);
        border: 1px solid rgba(255,255,255,0.1);
    }

    /* Input box */
    textarea {
        background: rgba(30, 41, 59, 0.7) !important;
        color: white !important;
        border-radius: 12px !important;
        border: 1px solid rgba(255,255,255,0.1) !important;
    }

    /* Buttons */
    button {
        border-radius: 12px !important;
        background: linear-gradient(135deg, #6366f1, #22c55e) !important;
        color: white !important;
        border: none !important;
        transition: 0.3s;
    }

    button:hover {
        transform: scale(1.05);
        box-shadow: 0px 0px 20px rgba(99,102,241,0.6);
    }

    /* Logs panel */
    .log-box {
        background: rgba(2, 6, 23, 0.8);
        border-radius: 16px;
        padding: 10px;
        border: 1px solid rgba(255,255,255,0.1);
        height: 500px;
        overflow-y: auto;
    }

    h1 {
        background: linear-gradient(90deg, #22c55e, #6366f1);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        text-align: center;
    }

    """
) as demo:

    gr.Markdown("# ⚖️ Legal AI")
    gr.Markdown("### 🔥 AIR RAG • LoRA Intelligence • Supreme Court Knowledge")

    with gr.Row():

        # LEFT SIDE
        with gr.Column(scale=2):

            chatbot = gr.Chatbot(height=520)

            user_input = gr.Textbox(
                placeholder="Ask anything about Supreme Court cases...",
                lines=2
            )

            with gr.Row():
                submit_btn = gr.Button("🚀 Ask")
                clear_btn = gr.Button("🧹 Clear")

        # RIGHT SIDE
        with gr.Column(scale=1):

            log_box = gr.Markdown(
                "<div class='log-box'>🧠 Waiting for query...</div>"
            )

    submit_btn.click(
        fn=process_legal_query,
        inputs=[user_input, chatbot],
        outputs=[chatbot, log_box]
    )

    clear_btn.click(
        lambda: ([], "<div class='log-box'>🧠 Logs cleared.</div>"),
        outputs=[chatbot, log_box]
    )

# ==============================
# LAUNCH
# ==============================
demo.queue().launch(share=True)

/tmp/ipykernel_55/1286595026.py:63: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_55/1286595026.py:63: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_55/1286595026.py:133: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=520)
/tmp/ipykernel_55/1286595026.py:133: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False i

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://ba2634c3fd11400b2d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

In [11]:
print(type(vectorstore.index))

<class 'faiss.swigfaiss.GpuIndexFlat'>


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
